In [20]:
suppressWarnings(suppressMessages({
    library("tidyverse")     
    library("seriation")     # OLO ordering of hclust
    library("dendextend")    # more visualization of dendrogram
    library("factoextra")    # more visualization of dendrogram
    library("FactoMineR")    # more visualization of dendrogram
    library("pheatmap")      # beautiful heatmap
    library("RColorBrewer")  # beautiful color
    library("IRdisplay")     # timely report in loops
    options(width = 160)
}))

## read test results

## Batch

In [23]:
files = c(
"heatmap_dist_continue_branching_2023_0914_2337_0125_mut",
"heatmap_dist_continue_branching_2023_0914_2337_0250_mut",
"heatmap_dist_continue_branching_2023_0914_2337_0500_mut",
"heatmap_dist_continue_branching_2023_0914_2337_1000_mut",
"heatmap_dist_continue_branching_2023_0914_2337_2000_mut",
"heatmap_dist_continue_branching_2023_0914_2337_0inf_mut"
)

In [24]:
for (runname in files){
    in_file <- paste0(runname, ".csv")
    out_fig <- paste0("output/", runname, ".pdf")
    dist_mtx <- read.csv(in_file, sep=",", row.names=1)

    # get heatmap color ready
    n_breaks = 100
    colors <- data.frame(CN=c(0, 0.1, 0.2, 0.5, 0.8, 0.9, 1),    # color intercepts
                         col=c("#000003", "#4393c3", "#80cdc1", "#ffffff", "#fee090", "#f46d43", "#67001f"))  # boarder line color
    col_brks <- c()
    for (i in 2:dim(colors)[1]){
        col_brks <- c(col_brks, seq(from=colors[i-1,"CN"], to=colors[i,"CN"], length.out=n_breaks))}
    col_brks <- unique(col_brks)
    cell_col <- colorRampPalette(colors$col)(length(col_brks))

    bin_hist <- hist(do.call(c, matrix(dist_mtx)), breaks = seq(0, 1, length.out = 100), plot = FALSE)
    bin_hist$counts <- log10(bin_hist$counts+1)
    
    bin_mids <- bin_hist$mids
    for (i in 1:length(bin_mids)) { bin_mids[i] <- which.min(abs(col_brks-bin_mids[i])) }
    bin_cols <- cell_col[bin_mids]
    
    try(dev.off(), silent=TRUE)
    pdf(file=out_fig, width=12, height=12)
    out_fig <- paste0("output/", runname, ".pdf")
    pheatmap(dist_mtx, 
             scale="none", color=cell_col, breaks=col_brks, # legend_breaks=c(0,1,2,5,10,20),
             show_rownames=TRUE, show_colnames=TRUE,        # this must be true
             #annotation_col = annot, annotation_colors = annot_color
             cluster_rows=FALSE, cluster_cols=FALSE, border_color = NA,
             clustering_callback=cb_sp_wrd, treeheight_col=120, cutree_col=10)
    try(dev.off(), silent=TRUE)
    out_fig <- paste0("output/", runname, ".png")
    png(file=out_fig, width=1200, height=1200)
    pheatmap(dist_mtx, 
             scale="none", color=cell_col, breaks=col_brks, # legend_breaks=c(0,1,2,5,10,20),
             show_rownames=TRUE, show_colnames=TRUE,        # this must be true
             #annotation_col = annot, annotation_colors = annot_color
             cluster_rows=FALSE, cluster_cols=FALSE, border_color = NA,
             clustering_callback=cb_sp_wrd, treeheight_col=120, cutree_col=10)
    try(dev.off(), silent=TRUE)
    }